# Self-attention and tile positions before ABMIL (RoFormer-MIL)

**The question:** does letting the slide's tiles attend to each other before
pooling, with or without knowing where they lie on the slide, beat our baseline
ABMIL, especially across hospitals?

The model tested is RoFormer-MIL's "Ro-ABMIL" (Pochet, Maroun and Trullo,
MICCAI MLMI workshop 2023): one transformer layer with 2D rotary position
embedding (RoPE) over the tiles, then ABMIL. RoPE rotates each tile's queries
and keys by its column and row on the tiling grid, so attention can depend on
how far apart two tiles are, and on nothing else about their positions. On
ResNet50 features the paper reports ABMIL 0.80 against Ro-ABMIL 0.90 AUROC on
Camelyon16; but in its own ablation a transformer layer without positions
scored 0.91 against RoPE's 0.92 (TCGA-NSCLC). So two arms are trained here:

| Arm | Model (`src/mil.py`) | What it adds over the baseline |
|---|---|---|
| baseline | `GatedABMIL` (notebook 12) | nothing: its saved predictions are reused |
| `transformer` | `TileTransformerABMIL(rope=False)` | self-attention between the slide's tiles |
| `roformer` | `TileTransformerABMIL(rope=True)` | the same, plus the tiles' relative positions |

`transformer` against the baseline asks whether self-attention helps;
`roformer` against `transformer` asks whether positions add anything beyond it.

**Everything else is the baseline's:** the packed tissue set
`uni2h-L1-224-t144-hsv-v1`, the same training recipe (AdamW 1e-4, weight decay
1e-5, one slide per step, cosine, 20 epochs, no early stopping), the same
protocols and seeds (mixed on the Stage 4 folds; every slide of one hospital to
every slide of the other, both directions, seeds 0 to 4) and the same decision
rule as notebook 14. The transformer layer (8 heads, MLP four times wider,
dropout 0.25) is fixed, not tuned; the paper does not give these values.

**One confound to keep in mind:** the two new arms have about four times the
baseline's parameters (4.2 M against 1.05 M). Between `transformer` and
`roformer` the count is equal, so that comparison is clean; against the
baseline, a gain could partly be capacity.

Positions come from `tiles.parquet`: level-0 coordinates divided by a tile's
level-0 size (896 px) and rounded. Nothing is re-extracted.

## Preamble

In [1]:
"""Make the project's src/ folder importable.

Locally it sits one folder up from notebooks/. On Kaggle it arrives as an
attached dataset, so every attached dataset is checked for a src/ folder, and
the notebook stops with a clear message if there is none.
"""

import glob
import os
import sys

SRC_ROOT = next(
    (
        root
        for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
        if os.path.isfile(os.path.join(root, "src", "tiling.py"))
    ),
    None,
)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

In [2]:
"""Imports.

functools, os, time   building each arm's model, paths, timing.
numpy, pandas         positions check, tables.
tqdm                  progress over folds and seeds.
src.features          the packed set, and where predictions are saved.
src.mil               the models, protocols, saved predictions and comparisons.
src.paths             where folds.csv is, locally and on Kaggle.
"""

import functools
import os
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from src.features import (
    open_packed,  # the packed set: features memory-mapped, index, starts, ends, tiles table with level-0 x, y
    predictions_file,  # where a model's predictions on a set are saved
)
from src.mil import (
    CROSS_SITE,  # the two cross-site directions: name -> (training hospital, test hospital)
    GRID_CELL,  # a tile's size in level-0 pixels (896): coordinates divided by it give grid positions
    TileTransformerABMIL,  # one transformer layer (rope=True: with 2D rotary positions) between embedding and ABMIL pooling
    compare_runs,  # two saved prediction tables of one protocol: QWKs, difference, paired bootstrap interval
    cross_site,  # every slide of one hospital to every slide of the other, once per seed
    cross_validate,  # the mixed protocol on the Stage 4 folds
    grade_distribution,  # share of slides per grade, true against predicted
    load_bags,  # every slide's features into RAM
    load_positions,  # every slide's tile grid positions (column, row), in packed order
    prediction_columns,  # the prediction columns of a saved table
    save_predictions,  # predictions to <version>/predictions/<name>.parquet
)
from src.paths import (
    DATA_DIR,  # folds.csv lives here locally
    find_input,  # finds folds.csv
)

In [3]:
"""Settings, labels and the packed set.

1. VERSION: the packed tissue set, the baseline's.
2. ARMS: name -> the function that builds a fresh model of that arm. A fresh
   model per fold and seed, created after seeding, as for the baseline.
3. PROTOCOLS: mixed and both cross-site directions, as notebook 12 saved them.
4. The run stops here if the baseline's predictions are missing (run notebook
   12 first).
"""

VERSION = "uni2h-L1-224-t144-hsv-v1"
ARMS = {
    "transformer": functools.partial(TileTransformerABMIL, rope=False),
    "roformer": functools.partial(TileTransformerABMIL, rope=True),
}
PROTOCOLS = ["mixed", *CROSS_SITE]

packed = open_packed(VERSION)
folds = pd.read_csv(find_input("folds.csv", local_dir=DATA_DIR)).set_index("slide_id")
labels = folds.loc[packed.index["slide_id"]]
missing = [p for p in PROTOCOLS if not os.path.exists(predictions_file(VERSION, f"abmil-{p}"))]
if missing:
    raise FileNotFoundError(f"baseline predictions missing for {missing}; run notebook 12 first")
print(f"{VERSION}: {len(labels):,} slides; arms {list(ARMS)}; protocols {PROTOCOLS}")

folds.csv: /home/aztrek/projects/Prostate-Cancer-Grading/data/folds.csv
uni2h-L1-224-t144-hsv-v1: 10,615 slides; arms ['transformer', 'roformer']; protocols ['mixed', 'radboud_to_karolinska', 'karolinska_to_radboud']


## Features and positions

Every slide's features into memory, and every tile's grid position. A check
first: how far each level-0 coordinate lies from its rounded grid cell. It must
be a few pixels at most (each slide's real downsample is slightly off 16), far
below half a cell (448 px), or the rounding would put tiles in the wrong cell.

In [4]:
"""Load features and positions; check the rounding to grid cells, and show how large the grids are."""

started = time.perf_counter()
bags = load_bags(packed)
positions = load_positions(packed)
print(f"loaded {len(bags):,} bags and positions in {time.perf_counter() - started:.0f} s")

xy = packed.tiles[["x", "y"]].to_numpy()
off_grid = np.abs(xy - np.rint(xy / GRID_CELL) * GRID_CELL)
print(f"largest distance from a grid cell: {off_grid.max()} px (half a cell is {GRID_CELL // 2} px)")
if off_grid.max() >= GRID_CELL // 4:
    raise ValueError("coordinates are not on the tiling grid")
span = pd.DataFrame({s: (p.max(0).values + 1).tolist() for s, p in positions.items()}, index=["columns", "rows"]).T
print(span.describe().round(1).loc[["50%", "max"]].to_string())

loaded 10,615 bags and positions in 1 s
largest distance from a grid cell: 26.0 px (half a cell is 448 px)
     columns  rows
50%     10.0  24.0
max    107.0  52.0


## Train and predict

Each arm under the three protocols: five folds for mixed, five seeds for each
cross-site direction. About 100 minutes per arm. Predictions are saved next to
the feature set as `predictions/<arm>-<protocol>.parquet`.

In [5]:
"""Train and predict every arm under every protocol; save the predictions."""

for arm, model_fn in ARMS.items():
    started = time.perf_counter()
    predicted = {"mixed": cross_validate(bags, labels, "mixed", progress=tqdm, model_fn=model_fn,
                                         positions=positions)}
    for name, (train_hospital, test_hospital) in CROSS_SITE.items():
        predicted[name] = cross_site(bags, labels, train_hospital, test_hospital, progress=tqdm,
                                     model_fn=model_fn, positions=positions)
    for protocol, values in predicted.items():
        save_predictions(VERSION, f"{arm}-{protocol}", labels, values)
    print(f"{arm}: trained in {(time.perf_counter() - started) / 60:.1f} min")

transformer: trained in 74.2 min
roformer: trained in 100.2 min


## Scores

Three comparisons, each on the same slides with a 95% paired bootstrap
interval (2,000 draws; for cross-site, five seeds averaged on every draw):
each arm against the baseline, and `roformer` against `transformer` (do
positions add anything?).

In [6]:
"""Compare each arm with the baseline, and roformer with transformer."""


def table(name, protocol):
    return pd.read_parquet(predictions_file(VERSION, f"{name}-{protocol}"))


pairs = [("abmil", "transformer"), ("abmil", "roformer"), ("transformer", "roformer")]
rows = []
for first, second in pairs:
    for protocol in PROTOCOLS:
        for row in compare_runs(table(first, protocol), table(second, protocol), protocol):
            rows.append({"comparison": f"{second} - {first}", **row})
comparison = pd.DataFrame(rows)
print(comparison.round(4).to_string(index=False))

            comparison              protocol     slides     n  first  second  second seed std  difference  95% low  95% high
   transformer - abmil                 mixed        all 10615 0.8643  0.8714           0.0000      0.0070  -0.0003    0.0141
   transformer - abmil                 mixed    radboud  5160 0.8240  0.8389           0.0000      0.0149   0.0031    0.0277
   transformer - abmil                 mixed karolinska  5455 0.8807  0.8779           0.0000     -0.0027  -0.0122    0.0064
   transformer - abmil radboud_to_karolinska        all  5455 0.6316  0.6617           0.0161      0.0300   0.0153    0.0444
   transformer - abmil karolinska_to_radboud        all  5160 0.7800  0.7508           0.0175     -0.0292  -0.0373   -0.0211
      roformer - abmil                 mixed        all 10615 0.8643  0.8740           0.0000      0.0097   0.0025    0.0173
      roformer - abmil                 mixed    radboud  5160 0.8240  0.8456           0.0000      0.0216   0.0083    0.0348


## Predicted grades

Share of slides per predicted grade, seeds pooled, against the truth, for each
cross-site direction.

In [7]:
"""Predicted-grade distribution per cross-site direction: true, baseline, then each arm."""

for protocol in CROSS_SITE:
    base = table("abmil", protocol)
    shares = {"true": grade_distribution(base["isup_grade"], base[prediction_columns(base)])["true"]}
    for name in ["abmil", *ARMS]:
        run = table(name, protocol).loc[base.index]
        shares["baseline" if name == "abmil" else name] = grade_distribution(
            run["isup_grade"], run[prediction_columns(run)])["predicted"]
    print(f"\n{protocol}: share of slides per grade")
    print(pd.DataFrame(shares).T.round(3).to_string())


radboud_to_karolinska: share of slides per grade
grade            0      1      2      3      4      5
true         0.353  0.333  0.122  0.058  0.088  0.046
baseline     0.386  0.253  0.055  0.162  0.094  0.051
transformer  0.395  0.248  0.040  0.162  0.115  0.039
roformer     0.428  0.254  0.054  0.131  0.098  0.036

karolinska_to_radboud: share of slides per grade
grade            0      1      2      3      4      5
true         0.187  0.165  0.131  0.179  0.149  0.189
baseline     0.207  0.151  0.139  0.096  0.296  0.112
transformer  0.190  0.109  0.154  0.135  0.245  0.166
roformer     0.186  0.114  0.175  0.132  0.240  0.154


## The decision rule

The same as notebook 14's, per comparison: better in both cross-site
directions (interval above zero) and not worse on mixed (interval not below
zero).

In [8]:
"""Apply the decision rule to each comparison and print the verdict with its numbers."""

for (first, second) in pairs:
    name = f"{second} - {first}"
    rows = comparison[(comparison["comparison"] == name) & (comparison["slides"] == "all")].set_index("protocol")
    cross_ok = all(rows.loc[p, "95% low"] > 0 for p in CROSS_SITE)
    mixed_ok = rows.loc["mixed", "95% high"] >= 0
    details = "; ".join(f"{p} {rows.loc[p, 'difference']:+.3f} [{rows.loc[p, '95% low']:+.3f}, "
                        f"{rows.loc[p, '95% high']:+.3f}]" for p in PROTOCOLS)
    print(f"{name:24s} {'helps' if cross_ok and mixed_ok else 'does not help':14s} {details}")

transformer - abmil      does not help  mixed +0.007 [-0.000, +0.014]; radboud_to_karolinska +0.030 [+0.015, +0.044]; karolinska_to_radboud -0.029 [-0.037, -0.021]
roformer - abmil         does not help  mixed +0.010 [+0.002, +0.017]; radboud_to_karolinska +0.087 [+0.073, +0.103]; karolinska_to_radboud -0.014 [-0.022, -0.006]
roformer - transformer   helps          mixed +0.003 [-0.004, +0.009]; radboud_to_karolinska +0.057 [+0.047, +0.068]; karolinska_to_radboud +0.015 [+0.010, +0.021]


## Result

Run 2026-11-18 on the RTX 4060 PC, packed tissue set `uni2h-L1-224-t144-hsv-v1`.
Every tile coordinate lay within 26 px of its grid cell (half a cell is 448 px).
Training took 74 minutes (`transformer`) and 100 minutes (`roformer`).

| Model | Mixed | Radboud to Karolinska (5 seeds) | Karolinska to Radboud (5 seeds) |
|---|---|---|---|
| ABMIL (baseline) | 0.864 | 0.632 (0.602 to 0.656) | 0.780 (0.775 to 0.784) |
| transformer layer + ABMIL | 0.871 | 0.662 (0.644 to 0.685) | 0.751 (0.732 to 0.773) |
| RoPE transformer layer + ABMIL | 0.874 | 0.719 (0.653 to 0.744) | 0.766 (0.753 to 0.777) |

Differences with 95% paired bootstrap intervals:

| Comparison | Mixed | Radboud to Karolinska | Karolinska to Radboud | Rule |
|---|---|---|---|---|
| transformer - ABMIL | +0.007 [-0.000, +0.014] | +0.030 [+0.015, +0.044] | -0.029 [-0.037, -0.021] | does not help |
| RoPE - ABMIL | +0.010 [+0.002, +0.017] | +0.087 [+0.073, +0.103] | -0.014 [-0.022, -0.006] | does not help |
| RoPE - transformer | +0.003 [-0.004, +0.009] | +0.057 [+0.047, +0.068] | +0.015 [+0.010, +0.021] | helps |

- **Positions help, given self-attention.** Adding RoPE to the same layer
  improves both cross-site directions beyond noise; that comparison has equal
  parameter counts, so it is clean.
- **Self-attention alone does not.** It gains from Radboud to Karolinska and
  loses the same amount the other way.
- **Against the baseline, RoPE gains on mixed (+0.010) and strongly from
  Radboud to Karolinska (+0.087, four of five seeds at 0.73 to 0.74), but
  loses 0.014 from Karolinska to Radboud**, so it fails the rule in one
  direction. The new arms have four times the baseline's parameters, so part of
  any gain over the baseline may be capacity.
- Predicted grades: the self-attention arms cut Karolinska to Radboud's excess
  of grade 4 (baseline 29.6%, true 14.9%) to about 24%, and raise grade 3 from
  9.6% to about 13% (true 17.9%).